# Ciclo de Vida de ML (CRISP-ML) - Ingeniería, Entrenamiento y Evaluación de Modelos

En esta fase entrenamos dos modelos de Machine Learning utilizando el conjunto de datos limpio y estructurado:
1. **Regresión Lineal**: Modelo de regresión para estimar el promedio de calificación final del estudiante.
2. **Regresión Logística**: Modelo de clasificación para predecir si el estudiante será promovido o reprobado.

### Paso 1: Carga de librerías de modelado
Importamos herramientas de modelado de `scikit-learn` y para persistir archivos `joblib`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score, confusion_matrix
import joblib
import os

print("Módulos de Scikit-Learn cargados correctamente.")

### Paso 2: Carga y Preparación de Datos
Cargamos el dataset agregado, mapeamos los grados académicos a valores numéricos continuos e inicializamos las variables del estudio.

In [ ]:
df = pd.read_csv('resultados_estudiantes.csv', sep=';')

grado_map = {
    'P1': 0, 'P2': 0, 'P3': 0,
    '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    '6': 6, '7': 7, '8': 8, '9': 9, '10': 10, '11': 11,
    'AC': 6
}
df['grado_num'] = df['grado'].astype(str).map(grado_map).fillna(6).astype(int)
df.head()

### Paso 3: MODELO 1 - Regresión Lineal (Predicción del Promedio de Notas)
Dividimos los datos en 70% entrenamiento y 30% prueba. Estandarizamos las características (`grado_num`, `total_faltas`, `total_refuerzos`, `total_materias`) y entrenamos el modelo.

In [ ]:
# Selección de características y objetivo
X_reg = df[['grado_num', 'total_faltas', 'total_refuerzos', 'total_materias']]
y_reg = df['promedio_nota']

# Partición de datos
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# Escalado de variables
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

# Ajuste del modelo
lr_model = LinearRegression()
lr_model.fit(X_train_reg_scaled, y_train_reg)

# Evaluación
y_pred_reg = lr_model.predict(X_test_reg_scaled)
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Error Cuadrático Medio (MSE): {mse:.4f}")
print(f"Raíz del MSE (RMSE): {np.sqrt(mse):.4f}")
print(f"Coeficiente de Determinación (R2): {r2:.4f}")

### Paso 4: MODELO 2 - Regresión Logística (Predicción de Promoción Escolar)
Clasificamos si un estudiante será promovido o no basándonos en variables académicas clave (`promedio_nota`, `total_faltas`, `total_refuerzos`, `materias_reprobadas`, `grado_num`).

In [ ]:
X_clf = df[['promedio_nota', 'total_faltas', 'total_refuerzos', 'materias_reprobadas', 'grado_num']]
y_clf = df['promovido_num']

# Partición estratificada para mantener proporción de clases
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, stratify=y_clf, random_state=42
)

# Escalado de variables
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# Ajuste del modelo
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_clf_scaled, y_train_clf)

# Evaluación
y_pred_clf = log_model.predict(X_test_clf_scaled)
y_pred_prob = log_model.predict_proba(X_test_clf_scaled)[:, 1]

print(f"Precisión General (Accuracy): {accuracy_score(y_test_clf, y_pred_clf):.4f}")
print(f"Área bajo la curva ROC (ROC-AUC): {roc_auc_score(y_test_clf, y_pred_prob):.4f}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test_clf, y_pred_clf))
print("\nReporte de Clasificación:")
print(classification_report(y_test_clf, y_pred_clf))

### Paso 5: Exportación de Modelos
Guardamos los modelos y escaladores entrenados en la raíz del proyecto para consumirlos en la aplicación de Streamlit.

In [ ]:
joblib.dump(lr_model, 'modelo_lineal.joblib')
joblib.dump(scaler_reg, 'scaler_lineal.joblib')
joblib.dump(log_model, 'modelo_logistico.joblib')
joblib.dump(scaler_clf, 'scaler_logistico.joblib')

print("Modelos serializados con joblib exitosamente.")